# 04 — Claims Analytical Base

## Objective
Build the claims analytical table of the project:

`claims_analytical_base.csv`

This table enriches each claim with:
- member attributes
- policy information
- provider characteristics

It will support:
- fraud / abuse detection
- pricing analysis
- claims mix analysis
- provider behavior analysis

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth


In [2]:
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR
from src.data.load_data import (
    load_insured_members,
    load_policies,
    load_providers,
    load_claims,
)

In [3]:
## Load source tables
members = load_insured_members()
policies = load_policies()
providers = load_providers()
claims = load_claims()

print("members:", members.shape)
print("policies:", policies.shape)
print("providers:", providers.shape)
print("claims:", claims.shape)

members: (4000, 32)
policies: (4000, 25)
providers: (180, 17)
claims: (44144, 36)


In [4]:
display(claims.head(3))
display(providers.head(3))

,claim_id,member_id,policy_id,provider_id,claim_date,claim_year,claim_month,service_type,service_category,diagnosis_group,...,duplicate_claim_flag,temporal_anomaly_flag,upcoding_suspected_flag,provider_inflation_flag,mismatch_dx_proc_flag,suspicious_claim_flag,fraud_pattern_type,member_archetype_snapshot,cost_expected_band,claim_outlier_score_latent
0,CLM000000350,MBR001287,POL001287,PRV00080,2024-01-01,2024,1,Diagnostics,Laboratory,General,...,0,0,0,0,0,0,NaN,Healthy Low User,Medium,0.124
1,CLM000000547,MBR001823,POL001823,PRV00129,2024-01-01,2024,1,Diagnostics,Diagnostics,Preventive,...,0,0,0,0,0,0,NaN,Healthy Low User,Low,0.061
2,CLM000000911,MBR003002,POL003002,PRV00110,2024-01-01,2024,1,Outpatient,Follow-up Consultation,Minor Acute,...,0,0,0,0,0,0,NaN,Healthy Low User,Low,0.068


,provider_id,provider_name,provider_type,specialty_group,region,city_tier,network_status,provider_archetype,contract_type,base_cost_multiplier,diagnostic_intensity,admission_intensity,average_claim_expected,historical_volume_band,historical_suspicion_flag,provider_quality_proxy,fraud_exposure_score
0,PRV00001,Provider 1,Clinic,Pediatrics,Central,Metro,In-network,Normal,External,1.003462,Medium,Medium,209.30,High,0,High,0.177
1,PRV00002,Provider 2,Specialist Center,Pulmonology,Asunción,Semi-Urban,In-network,Normal,Standard,1.082438,Medium,Low,258.15,Low,0,Medium,0.238
2,PRV00003,Provider 3,Pharmacy,Pharmacy,Asunción,Urban,In-network,High Complexity,Standard,1.577247,Medium,Medium,319.41,Low,0,Medium,0.252


In [5]:
## Inspect claim columns and key candidates
print(claims.columns.tolist())
candidate_keys = [c for c in ["claim_id", "member_id", "policy_id", "provider_id"] if c in claims.columns]
candidate_keys

['claim_id', 'member_id', 'policy_id', 'provider_id', 'claim_date', 'claim_year', 'claim_month', 'service_type', 'service_category', 'diagnosis_group', 'diagnosis_severity', 'procedure_code_group', 'procedure_complexity', 'emergency_flag', 'inpatient_flag', 'surgery_flag', 'pharmacy_flag', 'preventive_flag', 'chronic_related_flag', 'maternity_related_flag', 'claim_amount_billed', 'claim_amount_approved', 'claim_amount_rejected', 'out_of_pocket_estimated', 'claim_processing_days', 'approval_rate', 'duplicate_claim_flag', 'temporal_anomaly_flag', 'upcoding_suspected_flag', 'provider_inflation_flag', 'mismatch_dx_proc_flag', 'suspicious_claim_flag', 'fraud_pattern_type', 'member_archetype_snapshot', 'cost_expected_band', 'claim_outlier_score_latent']


['claim_id', 'member_id', 'policy_id', 'provider_id']

In [6]:
## Prepare member-policy base
member_policy_base = members.merge(
    policies,
    on=["member_id", "policy_id"],
    how="left",
    validate="one_to_one"
)

print(member_policy_base.shape)
member_policy_base.head(3)

(4000, 55)


,member_id,policy_id,join_date,age,age_band,sex,region,city_tier,socioeconomic_band,employment_status,...,premium_monthly,premium_annual,underwriting_load_factor,discount_factor,recommended_plan_flag,pricing_adequacy_ratio,renewal_flag,cancellation_flag,sales_channel,broker_id
0,MBR000001,POL000001,2022-09-24,47,45-54,M,San Pedro,Urban,Upper-Middle,Self-Employed,...,291.4,3496.8,1.139,0.092,1,0.851,1,0,Corporate,NaN
1,MBR000002,POL000002,2024-04-02,35,35-44,F,San Pedro,Metro,Lower-Middle,Employed,...,198.7,2384.4,1.138,0.082,1,1.285,1,0,Direct,NaN
2,MBR000003,POL000003,2024-06-27,65,65+,M,Asunción,Urban,Middle,Self-Employed,...,430.1,5161.2,1.005,0.060,1,0.994,0,0,Direct,NaN


In [7]:
## Merge claims with member-policy and provider info
claims_analytical = claims.copy()

if "member_id" in claims_analytical.columns:
    claims_analytical = claims_analytical.merge(
        member_policy_base,
        on="member_id",
        how="left",
        suffixes=("", "_member")
    )

if "provider_id" in claims_analytical.columns:
    claims_analytical = claims_analytical.merge(
        providers,
        on="provider_id",
        how="left",
        suffixes=("", "_provider")
    )

print("claims_analytical shape:", claims_analytical.shape)
claims_analytical.head(3)

claims_analytical shape: (44144, 106)


,claim_id,member_id,policy_id,provider_id,claim_date,claim_year,claim_month,service_type,service_category,diagnosis_group,...,provider_archetype,contract_type,base_cost_multiplier,diagnostic_intensity,admission_intensity,average_claim_expected,historical_volume_band,historical_suspicion_flag,provider_quality_proxy,fraud_exposure_score
0,CLM000000350,MBR001287,POL001287,PRV00080,2024-01-01,2024,1,Diagnostics,Laboratory,General,...,Normal,Standard,0.892698,Low,Medium,197.96,High,0,High,0.197
1,CLM000000547,MBR001823,POL001823,PRV00129,2024-01-01,2024,1,Diagnostics,Diagnostics,Preventive,...,Expensive,Standard,1.552278,Low,Medium,388.66,Low,0,Low,0.292
2,CLM000000911,MBR003002,POL003002,PRV00110,2024-01-01,2024,1,Outpatient,Follow-up Consultation,Minor Acute,...,Normal,Preferred,1.044202,Medium,Medium,201.38,High,0,Low,0.227


In [8]:
## Basic validation after merges
validation_rows = []

if "claim_id" in claims.columns:
    validation_rows.append({
        "check": "claim_id rows preserved",
        "result": len(claims_analytical) == len(claims)
    })
    validation_rows.append({
        "check": "claim_id uniqueness preserved",
        "result": claims_analytical["claim_id"].nunique() == claims["claim_id"].nunique()
    })

if "member_id" in claims.columns:
    validation_rows.append({
        "check": "member profile merged",
        "result": "age" in claims_analytical.columns
    })

if "provider_id" in claims.columns:
    validation_rows.append({
        "check": "provider profile merged",
        "result": "provider_type" in claims_analytical.columns
    })

pd.DataFrame(validation_rows)

,check,result
0,claim_id rows preserved,True
1,claim_id uniqueness preserved,True
2,member profile merged,True
3,provider profile merged,True


In [9]:
## Date handling
date_candidates = [c for c in claims_analytical.columns if "date" in c.lower()]
date_candidates

['claim_date', 'join_date', 'policy_start_date', 'policy_end_date']

In [10]:
for col in date_candidates:
    claims_analytical[col] = pd.to_datetime(claims_analytical[col], errors="coerce")

claims_analytical[date_candidates].head(3) if date_candidates else print("No date columns found")

,claim_date,join_date,policy_start_date,policy_end_date
0,2024-01-01,2022-08-21,2022-08-21,2026-12-31
1,2024-01-01,2022-09-09,2022-09-09,2026-12-31
2,2024-01-01,2022-08-02,2022-08-02,2026-12-31


In [11]:
## Create first derived variables
# Monetary candidates
amount_candidates = [c for c in claims_analytical.columns if any(x in c.lower() for x in ["amount", "cost", "paid", "claim"])]
amount_candidates[:20]
# Create generic derived variables when source columns are available
if "claim_date" in claims_analytical.columns:
    claims_analytical["claim_year"] = claims_analytical["claim_date"].dt.year
    claims_analytical["claim_month"] = claims_analytical["claim_date"].dt.to_period("M").astype(str)

if "premium_monthly" in claims_analytical.columns and "premium_annual" in claims_analytical.columns:
    claims_analytical["premium_gap"] = claims_analytical["premium_annual"] - claims_analytical["premium_monthly"] * 12

# Try to identify a paid/cost variable
cost_col = None
for candidate in ["claim_amount", "paid_amount", "approved_amount", "amount_paid", "cost_amount", "total_cost"]:
    if candidate in claims_analytical.columns:
        cost_col = candidate
        break

print("Detected cost column:", cost_col)
if cost_col is not None:
    claims_analytical["high_cost_claim_flag"] = (
        claims_analytical[cost_col] >= claims_analytical[cost_col].quantile(0.95)
    ).astype(int)

    if "average_claim_expected" in claims_analytical.columns:
        claims_analytical["claim_vs_expected_ratio"] = (
            claims_analytical[cost_col] / claims_analytical["average_claim_expected"]
        ).replace([np.inf, -np.inf], np.nan)

        claims_analytical["high_claim_vs_expected_flag"] = (
            claims_analytical["claim_vs_expected_ratio"] >= claims_analytical["claim_vs_expected_ratio"].quantile(0.95)
        ).astype(int)

Detected cost column: None


In [12]:
## Missingness in enriched claims table
missing_claims = pd.DataFrame({
    "column": claims_analytical.columns,
    "missing_n": claims_analytical.isna().sum().values,
    "missing_pct": (claims_analytical.isna().sum().values / len(claims_analytical)).round(4)
}).sort_values(["missing_pct", "missing_n"], ascending=[False, False])

missing_claims.head(20)

,column,missing_n,missing_pct
32,fraud_pattern_type,39813,0.9019
89,broker_id,24999,0.5663
54,chronic_group,19962,0.4522
0,claim_id,0,0.0000
1,member_id,0,0.0000
2,policy_id,0,0.0000
3,provider_id,0,0.0000
4,claim_date,0,0.0000
5,claim_year,0,0.0000
6,claim_month,0,0.0000


In [13]:
## Quick business profiling
profile_rows = []

if "provider_type" in claims_analytical.columns:
    display(
        claims_analytical["provider_type"]
        .value_counts(dropna=False)
        .rename_axis("provider_type")
        .reset_index(name="n_claims")
    )

if "region" in claims_analytical.columns:
    display(
        claims_analytical["region"]
        .value_counts(dropna=False)
        .rename_axis("region")
        .reset_index(name="n_claims")
    )

,provider_type,n_claims
0,Clinic,21909
1,Hospital,12734
2,Pharmacy,3168
3,Specialist Center,2952
4,Lab,2151
5,Imaging,1230


,region,n_claims
0,Central,13104
1,Asunción,9298
2,Alto Paraná,6316
3,Itapúa,4355
4,Cordillera,3709
5,Caaguazú,2736
6,San Pedro,2402
7,Paraguarí,2224


In [14]:
if cost_col is not None:
    display(claims_analytical[cost_col].describe())

    if "provider_type" in claims_analytical.columns:
        display(
            claims_analytical.groupby("provider_type")[cost_col]
            .agg(["count", "mean", "median", "sum"])
            .sort_values("sum", ascending=False)
            .reset_index()
        )

In [15]:
## Save analytical base
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_DIR / "claims_analytical_base.csv"
claims_analytical.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\processed\claims_analytical_base.csv


In [16]:
## Final notes

In [17]:
notes = [
    "claims_analytical_base.csv is now the main claim-level table of the project.",
    "This table will feed fraud / abuse detection, provider analysis, and later pricing work.",
    "Next step: build provider_master.csv."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

1. claims_analytical_base.csv is now the main claim-level table of the project.
2. This table will feed fraud / abuse detection, provider analysis, and later pricing work.
3. Next step: build provider_master.csv.
